In [26]:
import stanza
import pandas as pd
import re
from IPython.display import display
import sys
import os
import json

In [19]:
if 'google.colab' in sys.modules: 
    if not os.path.exists('/content/nlp_course_project'):
        !git clone https://github.com/Karoshi-man/nlp_course_project.git
    
    %cd /content/nlp_course_project
    !pip install ftfy regex pandas stanza -q
    sys.path.append('/content/nlp_course_project')
    
    FOLDER_ID = '1pIDpBFJ33L9XrldgXEXiAnLRNCs6f0gb'
    
    os.makedirs('/content/nlp_course_project/data', exist_ok=True)
    !gdown --folder https://drive.google.com/drive/folders/{FOLDER_ID} -O /content/nlp_course_project/data/
    
    data_dir = '/content/nlp_course_project/data'

else:
    sys.path.append(os.path.abspath('..'))
    data_dir = '../data'

In [20]:
stanza.download('uk', processors='tokenize,pos,lemma')
nlp = stanza.Pipeline(lang='uk', processors='tokenize,pos,lemma', use_gpu=False)

2026-03-03 21:37:58 INFO: Downloaded file to C:\Users\mfese\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\resources.json
2026-03-03 21:37:58 WARNING: Language uk package default expects mwt, which has been added
2026-03-03 21:37:58 INFO: Downloading these customized packages for language: uk (Ukrainian)...
| Processor       | Package     |
---------------------------------
| tokenize        | iu          |
| mwt             | iu          |
| pos             | iu_charlm   |
| lemma           | iu_nocharlm |
| pretrain        | conll17     |
| forward_charlm  | conll17     |
| backward_charlm | conll17     |

2026-03-03 21:37:58 INFO: File exists: C:\Users\mfese\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\uk\tokenize\iu.pt
2026-03-03 21:37:58 INFO: File exists: C:\Users\mfese\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\uk\mwt\iu.pt
2026-03-03 21:37:58 INFO: File exists: C:\Users\mfese\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\uk\pos\iu_char

In [21]:
processed_data_path = os.path.join(data_dir, "processed_v2", "processed_v2.csv")

if os.path.exists(processed_data_path):
    df_processed = pd.read_csv(processed_data_path)
    display(df_processed.head(2))
else:
    print(f" File {processed_data_path} not found")

,title,company,link,source_category,description,clean_text,sentences
0,RnD Engineer (CV + ML),Warbirds,https://jobs.dou.ua/companies/warbirds-boiovi-...,AI/ML,Warbirds Бойові Птахи України Всі вакансії ком...,Warbirds Бойові Птахи України Всі вакансії ком...,['Warbirds Бойові Птахи України Всі вакансії к...
1,AI Video Creator,HOLYWATER TECH,https://jobs.dou.ua/companies/holy-water/vacan...,AI/ML,HOLYWATER TECH Всі вакансії компанії\r\nHOLYWA...,HOLYWATER TECH Всі вакансії компанії\nHOLYWATE...,['HOLYWATER TECH Всі вакансії компанії HOLYWAT...


In [ ]:
gold_subset_path = "tests/gold_subset_50.jsonl"
gold_subset = []

with open(gold_subset_path, "r", encoding="utf-8") as f:
    for line in f:
        gold_subset.append(json.loads(line.strip()))

EXPERIENCE_RULE = re.compile(
    r"(?:досвід[уа]?|experience|exp)[\s\w]{0,25}?(\d+(?:[.,]\d+)?\s*(?:\+|-)?\s*(?:до)?\s*(?:\d+)?(?:-х|х)?)\s*(?:рок|років|years|yrs|р\.)",
    re.IGNORECASE
)

results = []

for row in gold_subset:
    raw_text = row["raw_text"]
    doc = nlp(raw_text)
    lemma_text = " ".join([word.lemma if word.lemma else word.text for sent in doc.sentences for word in sent.words])
    match_raw = EXPERIENCE_RULE.search(raw_text)
    pred_raw = match_raw.group(1).strip() if match_raw else None
    match_lemma = EXPERIENCE_RULE.search(lemma_text)
    pred_lemma = match_lemma.group(1).strip() if match_lemma else None
    
    results.append({
        "Expected": row["expected_exp"],
        "Pred (Raw)": pred_raw,
        "Pred (Lemma)": pred_lemma
    })

In [30]:
df_results = pd.DataFrame(results)

correct_raw = sum(1 for r in results if r["Expected"] == r["Pred (Raw)"])
correct_lemma = sum(1 for r in results if r["Expected"] == r["Pred (Lemma)"])
total = len(results)

precision_raw = (correct_raw / total) * 100
precision_lemma = (correct_lemma / total) * 100

print(f"Оцінка на Gold Subset ({total} прикладів):")
print(f"Precision (Raw): {precision_raw:.1f}% ({correct_raw}/{total})")
print(f"Precision (Lemma): {precision_lemma:.1f}% ({correct_lemma}/{total})")

Оцінка на Gold Subset (50 прикладів):
Precision (Raw): 68.0% (34/50)
Precision (Lemma): 16.0% (8/50)


In [23]:
def get_lemma_text(text):
    doc = nlp(text)
    lemmas = []
    for sentence in doc.sentences:
        for word in sentence.words:
            lemmas.append(word.lemma if word.lemma else word.text)
    return " ".join(lemmas)

In [24]:
for row in gold_subset:
    row["lemma_text"] = get_lemma_text(row["raw_text"])

print(f"raw: {gold_subset[0]['raw_text']}")
print(f"lemma: {gold_subset[0]['lemma_text']}")

raw: Шукаємо крутого Senior Data Engineer з досвідом 3+ років.
lemma: шукати крутий Senior Data Engineer з досвід 3+ рік .


In [25]:
res = []

for row in gold_subset:
    match_raw = EXPERIENCE_RULE.search(row["raw_text"])
    pred_raw = match_raw.group(1).strip() if match_raw else None
    match_lemma = EXPERIENCE_RULE.search(row["lemma_text"])
    pred_lemma = match_lemma.group(1).strip() if match_lemma else None
    
    res.append({
        "Текст (Raw)": row["raw_text"],
        "Expected": row["expected_exp"],
        "Pred (Raw Baseline)": pred_raw,
        "Pred (Lemma Baseline)": pred_lemma
    })

df_res = pd.DataFrame(res)
display(df_res)

correct_raw = sum(1 for r in res if r["Expected"] == r["Pred (Raw Baseline)"])
correct_lemma = sum(1 for r in res if r["Expected"] == r["Pred (Lemma Baseline)"])

print(f"Baseline 1 (raw): {correct_raw} / {len(gold_subset)}")
print(f"Baseline 2 (lemma): {correct_lemma} / {len(gold_subset)}")

,Текст (Raw),Expected,Pred (Raw Baseline),Pred (Lemma Baseline)
0,Шукаємо крутого Senior Data Engineer з досвідо...,3+,3+,None
1,"Вимоги: комерційний досвід роботи від 1,5 року.","1,5","1,5",None
2,Розглядаємо кандидатів без досвіду роботи.,NaN,NaN,None
3,Бажаний досвід 2 роки в IT.,2,2,None
4,"Компанії вже 10 років на ринку, шукаємо мідла ...",5,5,None


Baseline 1 (raw): 5 / 5
Baseline 2 (lemma): 1 / 5


In [ ]:
edge_cases_path = "tests/ling_edge_cases.jsonl"

if os.path.exists(edge_cases_path):
    with open(edge_cases_path, "r", encoding="utf-8") as f:
        for line in f:
            case = json.loads(line.strip())
            
            doc = nlp(case["text"])
            lemmas = [word.lemma if word.lemma else word.text for sent in doc.sentences for word in sent.words]
            pos_tags = [word.upos for sent in doc.sentences for word in sent.words]
            
            print(f"Original: {case['text']}")
            print(f"Lemaa:    {' '.join(lemmas)}")
            print(f"POS:     {' '.join(pos_tags)}")
            print(f"problem:  {case['expected_issue']}")
            print("-" * 60)
else:
    print(f"File {edge_cases_path} not found")

Original: Шукаємо крутого сеньйора з досвідом.
Lemaa:    шукати крутий сеньйор з досвід .
POS:     VERB ADJ NOUN ADP NOUN PUNCT
problem:  Сленг ('сеньйора') може не лематизуватися або отримати хибний POS-тег.
------------------------------------------------------------
Original: Треба фіксити баги на бекенді.
Lemaa:    треба фіксити бага на бекенда .
POS:     ADV VERB NOUN ADP NOUN PUNCT
problem:  Суржик/Англіцизми ('фіксити', 'баги', 'бекенді') — чи знає їх Stanza?
------------------------------------------------------------
Original: Досвід роботи з Node.js та Vue.js обов'язковий.
Lemaa:    досвід робота з Node .js та Vue .js обов’язковий .
POS:     NOUN NOUN ADP X X CCONJ X X ADJ PUNCT
problem:  Крапки у назвах технологій. Stanza може розірвати 'Node.js' на леми 'Node' + '.' + 'js'.
------------------------------------------------------------
Original: Пишемо на C++ та C#.
Lemaa:    писати на C + + та C # .
POS:     VERB ADP X X SYM CCONJ X SYM PUNCT
problem:  Спецсимволи (+, #). Мо

In [28]:
summary_path = "docs/audit_summary_lab3.md"

markdown_content = """# Audit Summary Lab 3: Lemma & POS Tagging

## 1. Аналіз помилок (Error Analysis)
Використання бібліотеки Stanza для ІТ-домену (вакансії DOU) виявило наступні критичні проблеми:
* **Руйнування назв технологій:** Спецсимволи та крапки відокремлюються. `Node.js` → `Node . js`, `C++` → `C + +`. Це критично ламає витяг навичок (Skills/Tech).
* **Некоректна обробка сленгу та трансліту:** Англіцизми отримують хибні леми (`бекенді` → `бекенда`, `баги` → `бага`).
* **Втрата форматування числівників:** `1,5` розбивається на `1 , 5`, `4k` → `4 k`.

## 2. Baseline Порівняння
* **Baseline 1 (Raw Text + Regex):** Точність витягування досвіду — 100%.
* **Baseline 2 (Lemma Text + Regex):** Точність витягування досвіду — 20%.
* **Причина падіння:** Оригінальні регулярні вирази спираються на специфічні маркери, які після лематизації змінюють форму. Крім того, токенізація Stanza додає пробіли між цифрами та знаками, що повністю ламає контекстні вікна.

## 3. Висновок: Коли леми додають цінність
* **Рішення для проєкту:** У задачі Extraction / NER (витягування скілів, технологій, локацій) **леми НЕ використовуємо**, оскільки вони руйнують структуру іменованих сутностей та IT-термінів. 
* Дані продовжують свій шлях у пайплайні у стані `processed_v2`. Леми та POS-теги можуть бути залучені виключно як допоміжні фічі на етапах дедуплікації або повнотекстового пошуку (Retrieval).
"""

os.makedirs(os.path.dirname(summary_path), exist_ok=True)
with open(summary_path, "w", encoding="utf-8") as f:
    f.write(markdown_content)